In [10]:
import pandas as pd
from pathlib import Path

# =========================
# Paths
# =========================
BASE_DIR = Path("E:/IDEAL")
PROC_DIR = BASE_DIR / "processed"
OUT_DIR = PROC_DIR / "final"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 1) Load datasets
# =========================
hourly_path = PROC_DIR / "IDEAL_master_hourly.csv"
meta_path = PROC_DIR / "aggregated_metadata" / "home_metadata.csv"

hourly = pd.read_csv(hourly_path, parse_dates=["timestamp"])
meta = pd.read_csv(meta_path)

# =========================
# 2) Clean / standardize column names
# =========================
hourly.columns = hourly.columns.str.strip()
meta.columns = meta.columns.str.strip()

# =========================
# 3) Ensure we have the same join key name: home_id
# - hourly already has home_id (e.g. "home100")
# - meta has homeid numeric -> rename & convert to "homeXX"
# =========================
# Rename meta key to home_id if needed
if "homeid" in meta.columns and "home_id" not in meta.columns:
    meta = meta.rename(columns={"homeid": "home_id"})

# Cast to string + strip
hourly["home_id"] = hourly["home_id"].astype(str).str.strip()
meta["home_id"] = meta["home_id"].astype(str).str.strip()

# Convert meta home_id from "47" -> "home47" ONLY if it doesn't already start with "home"
meta["home_id"] = meta["home_id"].apply(lambda x: x if x.lower().startswith("home") else "home" + x)

# Safety: remove duplicate home_id rows in meta (should be none, but just in case)
meta = meta.drop_duplicates(subset=["home_id"], keep="first")

# =========================
# 4) Merge (many-to-one)
# =========================
data = hourly.merge(meta, on="home_id", how="left")

print("After join:")
print("Rows:", len(data))
print("Unique homes:", data["home_id"].nunique())

# =========================
# 5) Calendar features
# =========================
data["hour"] = data["timestamp"].dt.hour
data["day_of_week"] = data["timestamp"].dt.dayofweek # 0=Mon ... 6=Sun
data["is_weekend"] = data["day_of_week"].isin([5, 6]).astype(int)

# =========================
# 6) Scottish public holidays (2016-2018)
# =========================
SCOTTISH_HOLIDAYS = [
    # 2016
    "2016-01-01", "2016-01-04", "2016-03-25", "2016-03-28",
    "2016-05-02", "2016-05-30", "2016-08-01", "2016-11-30",
    "2016-12-26", "2016-12-27",
    # 2017
    "2017-01-02", "2017-01-03", "2017-04-14", "2017-04-17",
    "2017-05-01", "2017-05-29", "2017-08-07", "2017-11-30",
    "2017-12-25", "2017-12-26",
    # 2018
    "2018-01-01", "2018-01-02", "2018-03-30", "2018-04-02",
    "2018-05-07", "2018-05-28", "2018-08-06", "2018-11-30",
    "2018-12-25", "2018-12-26",
]

holidays = pd.to_datetime(SCOTTISH_HOLIDAYS)
data["is_holiday"] = data["timestamp"].dt.normalize().isin(holidays).astype(int)

# =========================
# 7) Sanity checks
# =========================
print("\nNaN check (core):")
check_cols = ["consumption_Wh", "internal_temperature", "external_temperature",
              "total_floor_area_m2", "residents"]
existing = [c for c in check_cols if c in data.columns]
print(data[existing].isna().sum())

match_rate = data["residents"].notna().mean() if "residents" in data.columns else None
if match_rate is not None:
    print("\nMatch rate (residents not null):", round(match_rate * 100, 2), "%")

# =========================
# 8) Save final dataset
# =========================
out_path = OUT_DIR / "IDEAL_final_hourly_dataset.csv"
data.to_csv(out_path, index=False)

print("\nSaved final dataset to:")
print(out_path.resolve())

data.head()

After join:
Rows: 1529350
Unique homes: 254

NaN check (core):
consumption_Wh          0
internal_temperature    0
external_temperature    0
total_floor_area_m2     0
residents               0
dtype: int64

Match rate (residents not null): 100.0 %

Saved final dataset to:
E:\IDEAL\processed\final\IDEAL_final_hourly_dataset.csv


,home_id,timestamp,consumption_Wh,internal_temperature,external_temperature,residents,hometype,build_era,location,urban_rural_class,...,num_elderly,num_working_adults,num_full_time_workers,num_part_time_workers,num_males,num_females,hour,day_of_week,is_weekend,is_holiday
0,home100,2017-03-07 14:00:00,233,14.7,8.9,1,flat,1919-1930,Midlothian,3+,...,0,1,0,0,1,0,14,1,0,0
1,home100,2017-03-07 15:00:00,677,13.3,8.1,1,flat,1919-1930,Midlothian,3+,...,0,1,0,0,1,0,15,1,0,0
2,home100,2017-03-07 16:00:00,567,12.7,7.3,1,flat,1919-1930,Midlothian,3+,...,0,1,0,0,1,0,16,1,0,0
3,home100,2017-03-07 17:00:00,312,12.5,6.4,1,flat,1919-1930,Midlothian,3+,...,0,1,0,0,1,0,17,1,0,0
4,home100,2017-03-07 18:00:00,209,12.4,4.8,1,flat,1919-1930,Midlothian,3+,...,0,1,0,0,1,0,18,1,0,0
